In [39]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

In [40]:
df = pd.read_csv('train.txt', sep=';', header=None, names=['text', 'emotion'])

In [41]:
df.head()

,text,emotion
0,i didnt feel humiliated,sadness
1,i can go from feeling so hopeless to so damned...,sadness
2,im grabbing a minute to post i feel greedy wrong,anger
3,i am ever feeling nostalgic about the fireplac...,love
4,i am feeling grouchy,anger


In [42]:
df.isnull().sum()

text       0
emotion    0
dtype: int64

In [43]:
df.shape

(16000, 2)

In [44]:
df['emotion'].value_counts()

emotion
joy         5362
sadness     4666
anger       2159
fear        1937
love        1304
surprise     572
Name: count, dtype: int64

In [45]:
unique_emotions = df['emotion'].unique()
emotions_numbers = {}
i = 0
for emotion in unique_emotions:
    emotions_numbers[emotion] = i
    i += 1
df['emotion'] = df['emotion'].map(emotions_numbers)

In [46]:
df.head()

,text,emotion
0,i didnt feel humiliated,0
1,i can go from feeling so hopeless to so damned...,0
2,im grabbing a minute to post i feel greedy wrong,1
3,i am ever feeling nostalgic about the fireplac...,2
4,i am feeling grouchy,1


In [47]:
df['text'] = df['text'].str.lower()

In [48]:
import string
def remove_punctuation(text):
    return text.translate(str.maketrans('', '', string.punctuation))

df['text'] = df['text'].apply(remove_punctuation)

In [49]:
def remove_numbers(text):
    return ''.join([i for i in text if not i.isdigit()])

df['text'] = df['text'].apply(remove_numbers)

In [50]:
df.head()

,text,emotion
0,i didnt feel humiliated,0
1,i can go from feeling so hopeless to so damned...,0
2,im grabbing a minute to post i feel greedy wrong,1
3,i am ever feeling nostalgic about the fireplac...,2
4,i am feeling grouchy,1


In [51]:
def remove_urls(text):
    return ' '.join([word for word in text.split() if not word.startswith('http')])

df['text'] = df['text'].apply(remove_urls)

In [52]:
def remove_emojis(text):
    new = ""
    for character in text:
        if character.isascii():
            new += character
    return new

df['text'] = df['text'].apply(remove_emojis)

In [53]:
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.downloader import download

In [54]:
download('punkt')
download('stopwords')
download('punkt_tab')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Rishi\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Rishi\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Rishi\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [55]:
stop_words = set(stopwords.words('english'))
len(stop_words)

198

In [56]:
def remove_stopwords(text):
    tokens = word_tokenize(text)
    filtered_tokens = [word for word in tokens if word not in stop_words]
    return ' '.join(filtered_tokens)

try:
    df['text'] = df['text'].apply(remove_stopwords)
except Exception as e:
    print(f"Error occurred: {e}")

In [57]:
df.head()

,text,emotion
0,didnt feel humiliated,0
1,go feeling hopeless damned hopeful around some...,0
2,im grabbing minute post feel greedy wrong,1
3,ever feeling nostalgic fireplace know still pr...,2
4,feeling grouchy,1


In [58]:
from sklearn.feature_extraction.text import CountVectorizer

In [74]:
vectorizer = CountVectorizer(ngram_range=(2, 2))
vectorizer_1gram = CountVectorizer(ngram_range=(1, 1), max_features=1000)
vectorizer_3gram = CountVectorizer(ngram_range=(3, 3), max_features=1000)
vectorizer_1to3gram = CountVectorizer(ngram_range=(1, 3), max_features=1000) # 1 to 3 grams all together

In [75]:
X_bag_of_words = vectorizer.fit_transform(df['text'])

In [76]:
print(f"Shape of the bag-of-words matrix: {X_bag_of_words.shape}")
print(f"Number of unique words (features): {len(vectorizer.get_feature_names_out())}")
vectorizer.get_feature_names_out()[56:77]  # Display the first 10 feature names

Shape of the bag-of-words matrix: (16000, 95337)
Number of unique words (features): 95337


array(['ability money', 'ability mourn', 'ability pull',
       'ability quickly', 'ability rhyme', 'ability segregate',
       'ability simply', 'ability step', 'ability think',
       'ability thrilled', 'ability understand', 'abit dull',
       'abit grouchy', 'abit hopeless', 'abit uncertain', 'able acquire',
       'able actually', 'able apart', 'able attend', 'able better',
       'able boost'], dtype=object)

In [77]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [78]:
tfidf_vectorizer = TfidfVectorizer()

In [81]:
X_tfidf = tfidf_vectorizer.fit_transform(df['text'])

In [84]:
print(f"Shape of the bag-of-words matrix: {X_bag_of_words.shape}")
print(f"Number of unique words (features): {len(vectorizer.get_feature_names_out())}")
print(f"Shape of the TF-IDF matrix: {X_tfidf.shape}")
print("First 10 feature names:", tfidf_vectorizer.get_feature_names_out()[56:77])
print("Matrix (first 10 rows):", X_tfidf[0:10])

Shape of the bag-of-words matrix: (16000, 95337)
Number of unique words (features): 95337
Shape of the TF-IDF matrix: (16000, 15042)
First 10 feature names: ['academic' 'academics' 'academy' 'acause' 'accelerated' 'accent'
 'accentuating' 'accept' 'acceptable' 'acceptance' 'acceptances'
 'accepted' 'accepting' 'accepts' 'access' 'accessaries' 'accessibility'
 'accessories' 'accessory' 'accident' 'accidentally']
Matrix (first 10 rows): <Compressed Sparse Row sparse matrix of dtype 'float64'
	with 67 stored elements and shape (10, 15042)>
  Coords	Values
  (0, 3585)	0.5951235084078971
  (0, 4886)	0.16379156905484632
  (0, 6368)	0.7867657412767967
  (1, 5633)	0.2472161491010906
  (1, 4890)	0.11315242650138915
  (1, 6282)	0.3391180698816026
  (1, 3175)	0.4728128292698571
  (1, 6280)	0.34329019535147487
  (1, 690)	0.25735593796737716
  (1, 12271)	0.2703838370526541
  (1, 1926)	0.41218505617295464
  (1, 916)	0.40055397920248415
  (2, 4886)	0.08634244868782456
  (2, 6491)	0.18407021588693162


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(df['text'], df['emotion'], test_size=0.2, random_state=42)

In [ ]:
df.shape

(16000, 2)

In [ ]:
count_vectorizer = CountVectorizer() # bag of words
tfidf_vectorizer = TfidfVectorizer()

In [89]:
x_train_count = count_vectorizer.fit_transform(X_train)

In [92]:

x_test_count = count_vectorizer.transform(X_test)

In [93]:
from sklearn.naive_bayes import MultinomialNB

In [94]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [95]:
nb_model = MultinomialNB()

In [96]:
nb_model.fit(x_train_count, y_train)

,"alpha alpha: float or array-like of shape (n_features,), default=1.0Additive (Laplace/Lidstone) smoothing parameter(set alpha=0 and force_alpha=True, for no smoothing).",1.0
,"force_alpha force_alpha: bool, default=TrueIf False and alpha is less than 1e-10, it will set alpha to1e-10. If True, alpha will remain unchanged. This may causenumerical errors if alpha is too close to 0... versionadded:: 1.2.. versionchanged:: 1.4 The default value of `force_alpha` changed to `True`.",True
,"fit_prior fit_prior: bool, default=TrueWhether to learn class prior probabilities or not.If false, a uniform prior will be used.",True
,"class_prior class_prior: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None
Name,Type,Value
"class_count_ class_count_: ndarray of shape (n_classes,)Number of samples encountered for each class during fitting. Thisvalue is weighted by the sample weight when provided.","ndarray[float64](6,)","[3720.,1732.,1008., 459.,1540.,4341.]"
"class_log_prior_ class_log_prior_: ndarray of shape (n_classes,)Smoothed empirical log probability for each class.","ndarray[float64](6,)","[-1.24,-2. ,-2.54,-3.33,-2.12,-1.08]"
"classes_ classes_: ndarray of shape (n_classes,)Class labels known to the classifier","ndarray[int64](6,)","[0,1,2,3,4,5]"
"feature_count_ feature_count_: ndarray of shape (n_classes, n_features)Number of samples encountered for each (class, feature)during fitting. This value is weighted by the sample weight whenprovided.","ndarray[float64](6, 13357)","[[1.,1.,0.,...,0.,0.,1.], [1.,0.,0.,...,0.,0.,0.], [0.,0.,0.,...,0.,1.,0.], [0.,0.,0.,...,0.,0.,0.], [1.,0.,0.,...,0.,0.,0.], [0.,0.,1.,...,1.,2.,0.]]"
"feature_log_prob_ feature_log_prob_: ndarray of shape (n_classes, n_features)Empirical log probability of featuresgiven a class, ``P(x_i|y)``.","ndarray[float64](6, 13357)","[[-10.07,-10.07,-10.76,...,-10.76,-10.76,-10.07], [ -9.6 ,-10.29,-10.29,...,-10.29,-10.29,-10.29], [-10.06,-10.06,-10.06,...,-10.06, -9.36,-10.06], [ -9.79, -9.79, -9.79,..., -9.79, -9.79, -9.79], [ -9.53,-10.22,-10.22,...,-10.22,-10.22,-10.22], [-10.91,-10.91,-10.21,...,-10.21, -9.81,-10.91]]"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,13357


In [97]:
nb_model.score(x_test_count, y_test)

0.7684375

In [98]:
y_pred = nb_model.predict(x_test_count)

In [100]:
accuracy = accuracy_score(y_test, y_pred)
print("bag of words accuracy:", accuracy)

bag of words accuracy: 0.7684375


In [102]:
x_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
x_test_tfidf = tfidf_vectorizer.transform(X_test)

In [103]:
nb2_model = MultinomialNB()
nb2_model.fit(x_train_tfidf, y_train)

,"alpha alpha: float or array-like of shape (n_features,), default=1.0Additive (Laplace/Lidstone) smoothing parameter(set alpha=0 and force_alpha=True, for no smoothing).",1.0
,"force_alpha force_alpha: bool, default=TrueIf False and alpha is less than 1e-10, it will set alpha to1e-10. If True, alpha will remain unchanged. This may causenumerical errors if alpha is too close to 0... versionadded:: 1.2.. versionchanged:: 1.4 The default value of `force_alpha` changed to `True`.",True
,"fit_prior fit_prior: bool, default=TrueWhether to learn class prior probabilities or not.If false, a uniform prior will be used.",True
,"class_prior class_prior: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None
Name,Type,Value
"class_count_ class_count_: ndarray of shape (n_classes,)Number of samples encountered for each class during fitting. Thisvalue is weighted by the sample weight when provided.","ndarray[float64](6,)","[3720.,1732.,1008., 459.,1540.,4341.]"
"class_log_prior_ class_log_prior_: ndarray of shape (n_classes,)Smoothed empirical log probability for each class.","ndarray[float64](6,)","[-1.24,-2. ,-2.54,-3.33,-2.12,-1.08]"
"classes_ classes_: ndarray of shape (n_classes,)Class labels known to the classifier","ndarray[int64](6,)","[0,1,2,3,4,5]"
"feature_count_ feature_count_: ndarray of shape (n_classes, n_features)Number of samples encountered for each (class, feature)during fitting. This value is weighted by the sample weight whenprovided.","ndarray[float64](6, 13357)","[[0.5 ,0.48,0. ,...,0. ,0. ,0.31], [0.48,0. ,0. ,...,0. ,0. ,0. ], [0. ,0. ,0. ,...,0. ,0.29,0. ], [0. ,0. ,0. ,...,0. ,0. ,0. ], [0.36,0. ,0. ,...,0. ,0. ,0. ], [0. ,0. ,0.3 ,...,0.26,0.7 ,0. ]]"
"feature_log_prob_ feature_log_prob_: ndarray of shape (n_classes, n_features)Empirical log probability of featuresgiven a class, ``P(x_i|y)``.","ndarray[float64](6, 13357)","[[ -9.65, -9.66,-10.05,...,-10.05,-10.05, -9.79], [ -9.41, -9.8 , -9.8 ,..., -9.8 , -9.8 , -9.8 ], [ -9.69, -9.69, -9.69,..., -9.69, -9.44, -9.69], [ -9.59, -9.59, -9.59,..., -9.59, -9.59, -9.59], [ -9.47, -9.77, -9.77,..., -9.77, -9.77, -9.77], [-10.14,-10.14, -9.87,..., -9.9 , -9.61,-10.14]]"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,13357


In [104]:
y_pred = nb2_model.predict(x_test_tfidf)

In [105]:
accuracy = accuracy_score(y_test, y_pred)
print("tfidf accuracy:", accuracy)

tfidf accuracy: 0.66125


In [106]:
from sklearn.linear_model import LogisticRegression

In [107]:
lr = LogisticRegression(max_iter=1000)
lr.fit(x_train_tfidf, y_train)

,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is '

In [108]:
y_pred = lr.predict(x_test_tfidf)

In [109]:
accuracy = accuracy_score(y_test, y_pred)
print("tfidf accuracy:", accuracy)

tfidf accuracy: 0.861875
